(course-enzyme-pdb-frontier)=

# Path B - Module 29: PDB Bioassemblies & AltLocs

Many high-resolution industrial enzymes are solved using X-ray crystallography. These files often contain **Alternate Locations (AltLocs)**, which are ghost-like representations of atoms that exist in more than one position in the crystal. PDB paths and PDB text share the native `PDBFileHandler`, so every native target receives the same normalized records and diagnostics.

In this module, you will learn to clean these ambiguities to have a physically consistent starting point for your PETase simulation.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load a structure with alternate locations from the Protein Data Bank
pdb_id = 'pdb_id:1BRS'
pdb_text = msm.convert(pdb_id, to_form='string:pdb_text')
molsys = msm.convert(pdb_text, to_form='molsysmt.MolSys')

### 1. Identifying the Disorder
Let's see if our system has atoms with multiple locations. These are problematic for most force fields.

In [2]:
alternate_locations = msm.get(molsys, alternate_location=True)
n_alternate_atoms = sum(len(entries) for entries in alternate_locations or [])

print(f"Found {n_alternate_atoms} atoms with multiple positions in the crystal.")

Found 2 atoms with multiple positions in the crystal.


### 2. Solving AltLocs
The function `solve_atoms_with_alternate_location()` activates the position with the highest occupancy for each canonical atom site and prefers label `A` when occupancies tie. All variants remain available in `Structures.alternate_location`.

In [3]:
print(msm.get(molsys, alternate_location=True))

msm.build.solve_atoms_with_alternate_location(molsys)

print("Alternate locations resolved without changing the topology atom count.")

[{'2686': {'location_id': array(['A', 'B'], dtype=object), 'atom_id': ['2687', '2688'], 'occupancy': array([0.5, 0.5]), 'coordinates': <Quantity([[3.2742 2.2579 0.1536]
 [3.2757 2.2571 0.1533]], 'nanometer')>, 'b_factor': <Quantity([0.2466 0.2467], 'nanometer ** 2')>}, '2687': {'location_id': array(['A', 'B'], dtype=object), 'atom_id': ['2689', '2690'], 'occupancy': array([0.5, 0.5]), 'coordinates': <Quantity([[3.1412 2.241  0.1076]
 [3.3396 2.192  0.2619]], 'nanometer')>, 'b_factor': <Quantity([0.2594 0.2596], 'nanometer ** 2')>}}]
Alternate locations resolved without changing the topology atom count.


### 3. Biological Assembly check
Is the PETase active as a monomer or a dimer? MolSysMT reads PDB `REMARK 350` symmetry operators into `Structures.bioassembly`, remaps them when atoms are extracted, and can generate the functional unit from those native instructions.

In [4]:
# Make the biological assembly
full_unit = msm.build.make_bioassembly(molsys)

msm.info(full_unit, element='chain')

index,id,name,n atoms,n groups,n components,molecule index,molecule type,entity index,entity name
0,A,A,864,108,1,[0],['protein'],[0],['protein 0']
1,D,D,693,87,2,[1],['protein'],[1],['protein 3']
2,A,A,145,145,145,"[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146]","['water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water']",[2],['water']
3,D,D,77,77,77,"[147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223]","['water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water', 'water']",[2],['water']


--- 

### 🏁 END OF PHASE 3: THE VIRTUAL LAB

**Congratulations!** You have completed the structural preparation of your industrial biocatalyst. You have:
1.  **Isolated** the PETase.
2.  **Audited** and **Repaired** the missing atoms.
3.  **Engineered** a new disulfide bridge for thermal stability.
4.  **Synthesized** a purification tag and a BHET substrate.
5.  **Solvated** and **Patched** the system for a professional report.

Now that your industrial model is ready, we enter **Phase 4: Data Analyst**, where we will measure if our engineering actually improved the enzyme's geometry.